# PhishGuard – Model lớn chấm sơ bộ bản nháp (Giai đoạn 2) · bản code `76633d7e`

Model chấm (mặc định **Qwen3-32B**, lớn hơn model đã sinh nháp) cho mỗi mẫu điểm 1–5 + lỗi + gợi ý sửa.
Người duyệt đọc kỹ mẫu điểm thấp trước (`review.html`); model chấm **không thay** người duyệt.

**Trước khi chạy**: *Accelerator* **GPU T4 x2**, *Internet* **On**. Không cần dataset (dữ liệu nhúng trong ô 2).
Ô 4 mặc định `LIMIT = 30` để **thử trước**; ổn rồi đặt `LIMIT = 0` và chạy lại ô 4.

Kết quả: `/kaggle/working/judged.jsonl` → tải về, chép vào `training/chat_data/data/`. **Chạy tiếp** sau khi bị
ngắt: *Add Input* → *Upload* file `judged.jsonl` đã tải về, ô 2 tự nạp lại.

In [ ]:
# Ô 1 – Cấu hình
MODEL = "Qwen/Qwen3-32B-AWQ"  # model chấm: nên LỚN HƠN model sinh nháp (Qwen3-14B) để bắt được lỗi của nó.
                              # vLLM báo thiếu bộ nhớ thì đổi "Qwen/Qwen3-14B-AWQ" (yếu hơn nhưng vẫn có ích).
TP = 2
WORKERS = 32
PORT = 8000

In [ ]:
# Ô 2 – Giải nén mã nguồn + dữ liệu (nhúng sẵn, bản 76633d7e) vào /kaggle/working/chat_data
import base64, io, os, shutil, zipfile

WORK = "/kaggle/working/chat_data"
PAYLOAD = "..."
zipfile.ZipFile(io.BytesIO(base64.b64decode(PAYLOAD))).extractall(WORK)  # ghi đè code cũ nếu có
for d, _, files in os.walk("/kaggle/input", followlinks=True):  # chạy tiếp: nạp judged.jsonl đã tải lên
    if "judged.jsonl" in files and not os.path.exists(f"{WORK}/data/judged.jsonl"):
        shutil.copy(os.path.join(d, "judged.jsonl"), f"{WORK}/data/judged.jsonl")
        print("Chạy tiếp từ", os.path.join(d, "judged.jsonl"))
print("Bản code 76633d7e:", sorted(os.listdir(WORK)), os.listdir(f"{WORK}/data"))

In [ ]:
# Ô 3 – Cài vLLM và bật server kiểu OpenAI (lần đầu mất ~5–10 phút: cài thư viện + tải model)
!pip install -q vllm
# vLLM kéo PyTorch bản CUDA mới, lệch với TorchAudio có sẵn của Kaggle → transformers nạp TorchAudio là lỗi.
# Không cần xử lý âm thanh nên gỡ đi.
!pip uninstall -y -q torchaudio
import torch
print("torch", torch.__version__, "| CUDA", torch.version.cuda, "| GPU dùng được:", torch.cuda.is_available())
assert torch.cuda.is_available(), "PyTorch không thấy GPU (driver Kaggle cũ hơn bản CUDA của PyTorch?) — gửi dòng trên"
import subprocess, time, urllib.request

log = open("/kaggle/working/vllm.log", "w")
server = subprocess.Popen(
    ["python", "-m", "vllm.entrypoints.openai.api_server", "--model", MODEL, "--port", str(PORT),
     "--tensor-parallel-size", str(TP), "--dtype", "half",          # T4 không có bfloat16
     "--max-model-len", "8192", "--gpu-memory-utilization", "0.92"],
    stdout=log, stderr=subprocess.STDOUT)
ready = False
for i in range(180):
    try:
        urllib.request.urlopen(f"http://localhost:{PORT}/v1/models", timeout=5)
        ready = True
        print(f"Server sẵn sàng sau {i * 10} giây"); break
    except Exception:
        if server.poll() is not None:
            break  # vLLM đã dừng: in log bên dưới
        time.sleep(10)
log.flush()
if not ready:
    print("".join(open("/kaggle/working/vllm.log", encoding="utf-8", errors="replace").readlines()[-60:]))
    raise RuntimeError("vLLM không chạy được — gửi 60 dòng log ở trên")

In [ ]:
# Ô 4 – Chấm (LIMIT = 30 để thử, rải đều cả file; ổn rồi đặt 0 và chạy lại ô này)
LIMIT = 30
ARGS = f"--base-url http://localhost:{PORT}/v1 --model {MODEL} --workers {WORKERS} --no-think"
if LIMIT:
    ARGS += f" --limit {LIMIT}"
!cd $WORK && python judge.py $ARGS
!cp $WORK/data/judged.jsonl /kaggle/working/judged.jsonl

In [ ]:
# Ô 5 – Phân bố điểm theo nhóm + xem vài mẫu điểm thấp
import json, collections, random
rows = [json.loads(l) for l in open(f"{WORK}/data/judged.jsonl", encoding="utf-8")]
by = collections.defaultdict(collections.Counter)
for r in rows:
    by[r["category"]][r["judge"]["score"]] += 1
for c, cnt in by.items():
    print(f"{c:14}", {s: cnt[s] for s in range(1, 6)})
low = [r for r in rows if r["judge"]["score"] <= 3]
for r in random.sample(low, min(3, len(low))):
    print("=" * 80, "\n", r["id"], "→", r["judge"])
    for m in r["messages"]:
        print(f"--- {m['role']}:\n{m['content'][:800]}")